In [ ]:
from pathlib import Path
from demiflow import data
from project import resolve_root
from preparation.visual_concepts.visual_concepts_pipeline import config, run_pipeline


In [ ]:
# 固定输入：筛选结束后明确填写 reviews 的提交版本；不自动跟随 latest。
RUN = False
RUN_NAME = 'visual_concepts_v1'
QID_SOURCE = {'uri': 'demiwtg/preparation/qid_concepts/datasets/qid_concepts.lance', 'version': 1}
QID_REVIEW_SOURCE = None
LEGACY_SOURCES = []
LEGACY_SELECTION_SOURCE = None
ALIGNMENT_SOURCE = None
CHOICE_SOURCE = None
RELATED_SOURCES = []
RESOURCE_SOURCES = []
CONFIG = None
if QID_REVIEW_SOURCE is not None or LEGACY_SOURCES:
    CONFIG = config(run=RUN_NAME, qid_source=QID_SOURCE if QID_REVIEW_SOURCE else None,
                    qid_review_source=QID_REVIEW_SOURCE, legacy_sources=LEGACY_SOURCES,
                    legacy_selection_source=LEGACY_SELECTION_SOURCE, alignment_source=ALIGNMENT_SOURCE,
                    choice_source=CHOICE_SOURCE, related_sources=RELATED_SOURCES,
                    resource_sources=RESOURCE_SOURCES)


In [ ]:
# 本格 RUN=True 才写本模块表；零模型调用。
if RUN:
    if CONFIG is None:
        raise ValueError('先配置实际固定来源')
    result = run_pipeline(CONFIG)
    print(result)
else:
    print('未执行；配置固定来源后再开启 RUN。')


In [ ]:
# 独立只读：新 kernel 可单独运行。本格不写表、不调用模型。
import json
from pathlib import Path
from demiflow import data
from project import resolve_root
from IPython.display import display
ROOT = resolve_root()
VIEW_RUN = 'visual_concepts_v1'
summary_path = ROOT / 'demiwtg/preparation/visual_concepts/datasets' / f'summary__{VIEW_RUN}.lance'
if not summary_path.exists():
    print('尚无已提交摘要；不代表其他运行已完成。')
else:
    import lance
    summary_version = lance.dataset(str(summary_path)).version
    summaries = data.read_lance(str(summary_path), version=summary_version).take_all()
    display(summaries)
    ref = summaries[0]['output']
    # 限量完整展开前五条的名称候选；原始资料可按 concept_id 单独投影查询。
    display(data.read_lance(str(ROOT / ref['uri']), version=ref['version'], limit=5,
                columns=['concept_id', 'qid', 'canonical_name', 'canonical_name_candidates',
                         'task_status', 'conflict_fields']).take_all())
